# 분류 과제 — 탑승 항구 예측 (다중 분류)

이번 과제는 두 부분입니다.

**Part 1. 이론 복습** — 세션에서 배운 내용을 여러분의 말로 정리해보세요.
**Part 2. 코드 해석 & 모델 개선** — 주어진 코드를 실행하고, 결과를 읽고 해석해주세요.

> 코드는 모두 완성되어 있습니다. 위에서부터 실행하면서 **🤔 질문 셀에 답을 마크다운으로 작성**해주세요.
> 검색해서 베낀 문장보다, **어설퍼도 본인 말로 쓴 설명**이 훨씬 좋은 답안입니다.

---


---
# Part 1. 이론 복습

### 🤔 T1. 지도학습과 비지도학습의 차이를 설명해주세요. 

(각각의 예시도 하나씩 들어주면 좋습니다)

**답:**
지도학습이란 정답이 주어진 상태에서 학습하는 알고리즘이다. 예를 들면 어떤 사진이 주어졌을 때 사과인지 딸기인지 구분하기 위해서 미리 많은 딸기와 사과의 사진을 통해 정답을 맞추기 위해 연습시킵니다. 그리고 다음에 사진이 들어오면 딸기인지 사과인지 알아맞출 수 있게 한다.

비지도 학습은 정답이 주어지지 않은 상태에서 학습하는 알고리즘으로, 위와 같이 사과와 딸기라고 예를 들면 그들의 특징을 구분해서 알아서 두 부류로 구분해가는 걸 의미한다. 



### 🤔 T2. 회귀와 분류의 차이를 설명해주세요.  

둘 다 지도학습인데, 무엇이 다른가요?

**답:** 회귀는 연속형 변수를 예측하기 위해 사용되며, 분류는 범주형 변수를 예측하기 위해 사용된다.

### 🤔 T3. 이진 분류와 다중 분류의 차이를 설명해주세요

**답:** 이진분류는 어떤 변수가 참 또는 거짓만의 값을 가질 때이고, 다중분류는 가질 수 있는 값이 3개 이상일떄이다!



### 🤔 T4. 세션에서 배운 네 가지 분류 모델을 각각 한두 문장으로 설명해주세요.  

- 로지스틱 회귀
- 의사결정나무
- SVM
- kNN

**답:**

로지스틱 회귀: 이진 분류 모델을 푸는 대표적인 알고리즘으로, 샘플이 특정 클래스에 속할 확률을 추정하여 기준치에 따라 분류하는 것을 목표로 함. 로지스틱 함수를 적용해 출력값을 0에서 1사이로 변환해주는 걸 의미

의사결정나무: 조건에 따라 데이터를 분류하며, 최종적으로 데이터가 순수한 레이블의 집합으로 구성될 때까지 분류를 반복하는 분석방법

SVM: 클래스를 분류할 수 있는 다양한 경계선 중 최적의 라인을 찾아내는 알고리즘. 데이터를 분리하는 초평면 중에서 데이터들과 가장 거리가 먼 초평면을 선택하여 분리하는 지도 학습 기반의 이진 선형 분류 모델

KNN: 비슷한 특성을 가진 데이터끼리 서로 가까이 있는다는 점을 이용한 분류 알고리즘. 데이터로부터 거리가 가까운 k개의 다른 데이터 레이블을 참조하여 분류하는 알고리즘



---
# Part 2. 코드 해석 — 탑승 항구 예측 모델

지난 실습에서는 **생존 여부(Survived)** 를 예측하는 **이진 분류** 모델을 만들었습니다.
이번에는 같은 데이터로 **탑승 항구(Embarked)** 를 예측하는 **다중 분류** 모델을 만듭니다.

- 예측 대상: `Embarked` (S / C / Q)
- 주어지는 정보: 생존 여부, 객실 등급, 성별, 나이 등

## 1. 전처리

전처리 과정은 지난 실습과 거의 같습니다. 다만 **`Embarked`가 이제 예측 대상**이라는 점이 다릅니다.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("./titanic.csv")

df['Initial'] = ""
for index, row in df.iterrows():
    df.loc[index, 'Initial'] = row['Name'].split(',')[1].split('.')[0].strip()

df['Initial'] = df['Initial'].replace(
    ['Mlle', 'Mme', 'Ms', 'Dr', 'Major', 'Lady', 'Countess', 'Jonkheer', 'Col',
     'Rev', 'Capt', 'Sir', 'Don', 'the Countess'],
    ['Miss', 'Miss', 'Miss', 'Mr', 'Mr', 'Mrs', 'Mrs', 'Other', 'Other',
     'Other', 'Mr', 'Mr', 'Mr', 'Other'])

df.loc[(df.Age.isnull()) & (df.Initial == 'Mr'),     'Age'] = 33
df.loc[(df.Age.isnull()) & (df.Initial == 'Mrs'),    'Age'] = 36
df.loc[(df.Age.isnull()) & (df.Initial == 'Master'), 'Age'] = 5
df.loc[(df.Age.isnull()) & (df.Initial == 'Miss'),   'Age'] = 22
df.loc[(df.Age.isnull()) & (df.Initial == 'Other'),  'Age'] = 46

df.dropna(subset=['Embarked'], inplace=True)
df.drop(['Cabin', 'Name', 'PassengerId', 'Ticket'], axis=1, inplace=True)

df_org = df.copy()

df['Relatives'] = df["SibSp"] + df["Parch"]
df['Sex']       = df['Sex'].map({'male': 0, 'female': 1})
df['Age']       = (df['Age'] // 10).astype(int)
df['Fare']      = pd.qcut(df['Fare'], q=9, labels=range(9))
df['Embarked']  = df['Embarked'].map({'S': 1, 'C': 2, 'Q': 3})
df['Initial']   = df['Initial'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Other': 4})

print("데이터 크기:", df.shape)
df.head()

데이터 크기: (889, 10)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Initial,Relatives
0,0,3,0,2,1,0,0,1,0,1
1,1,1,1,3,1,0,7,2,2,1
2,1,3,1,2,0,0,2,1,1,0
3,1,1,1,3,1,0,7,1,2,1
4,0,3,0,3,0,0,2,1,0,0


### 🤔 Q1. 결측치 처리 순서  

`Embarked`는 이번 과제에서 **예측 대상(y)** 입니다. 그런데 코드를 보면 결측치가 있는 2개 행을 `dropna()`로 **지워버립니다.**

1. 예측 대상에 결측치가 있을 때, 왜 그 값을 추측해서 채우지 않고 **행 자체를 지울까요?**
2. 만약 `Embarked`의 결측치 2개를 최빈값인 `S`로 채워서 학습에 사용했다면 어떤 문제가 생길까요?


**답:**
1. y는 정답 레이블이기 때문에 추측해서 채우면 잘못된 정답을 학습시킬 수 있다. 
2. 최빈값이 S라고 결측치를 다 S로 대체하면 모델이 부정확해질 수 있다.......



## 2. target과 feature 분리

In [2]:
X = df.drop('Embarked', axis=1)
y = df['Embarked']

print("feature 목록:", list(X.columns))
print()
print("[ Embarked 분포 ]  (1=S, 2=C, 3=Q)")
print(y.value_counts().sort_index())
print()
print("가장 많은 클래스(S)의 비율: %.4f" % (y == 1).mean())

feature 목록: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Initial', 'Relatives']

[ Embarked 분포 ]  (1=S, 2=C, 3=Q)
Embarked
1    644
2    168
3     77
Name: count, dtype: int64

가장 많은 클래스(S)의 비율: 0.7244


## 3. 훈련/테스트 분리 및 스케일링

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("train:", len(X_train), " test:", len(X_test))
print()
print("[ y_test 분포 ]")
print(y_test.value_counts().sort_index())
print()
print("테스트 세트에서 S의 비율: %.4f" % (y_test == 1).mean())

train: 711  test: 178

[ y_test 분포 ]
Embarked
1    130
2     36
3     12
Name: count, dtype: int64

테스트 세트에서 S의 비율: 0.7303


In [4]:
def predict_embarked(model, scaler, survived, pclass, sex, age, sibsp, parch, fare, initial):
    input_data = pd.DataFrame({
        'Survived': [survived],
        'Pclass':   [pclass],
        'Sex':      [0 if sex == 'male' else 1],
        'Age':      [age // 10],
        'SibSp':    [sibsp],
        'Parch':    [parch],
        'Fare':     [fare],
        'Initial':  [0 if initial == 'Mr' else (1 if initial == 'Miss' else
                    (2 if initial == 'Mrs' else (3 if initial == 'Master' else 4)))],
        'Relatives': [sibsp + parch],
    })

    fare_bins = pd.qcut(df_org['Fare'], 9, retbins=True)[1]
    input_data['Fare'] = pd.cut(input_data['Fare'], bins=fare_bins,
                                labels=range(9), include_lowest=True)
    input_data = input_data[X.columns]
    scaled = scaler.transform(input_data)

    prediction = model.predict(scaled)[0]
    probability = model.predict_proba(scaled)[0][list(model.classes_).index(prediction)]

    return {1: 'S', 2: 'C', 3: 'Q'}[prediction], probability

## 4. 모델 1 — 로지스틱 회귀

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

lr_model = LogisticRegression()
lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)

print("로지스틱 회귀 모델의 정확도:", accuracy_score(y_test, lr_pred))
print()
print(classification_report(y_test, lr_pred, zero_division=0))

로지스틱 회귀 모델의 정확도: 0.7303370786516854

              precision    recall  f1-score   support

           1       0.73      1.00      0.84       130
           2       0.00      0.00      0.00        36
           3       0.00      0.00      0.00        12

    accuracy                           0.73       178
   macro avg       0.24      0.33      0.28       178
weighted avg       0.53      0.73      0.62       178



In [6]:
print("[ 모델이 실제로 예측한 값의 분포 ]")
print(pd.Series(lr_pred).value_counts())
print()
print("[ 혼동 행렬 (3x3) ]   행=실제, 열=예측")
print(pd.DataFrame(confusion_matrix(y_test, lr_pred),
                   index=['실제 S', '실제 C', '실제 Q'],
                   columns=['예측 S', '예측 C', '예측 Q']))

[ 모델이 실제로 예측한 값의 분포 ]
1    178
Name: count, dtype: int64

[ 혼동 행렬 (3x3) ]   행=실제, 열=예측
      예측 S  예측 C  예측 Q
실제 S   130     0     0
실제 C    36     0     0
실제 Q    12     0     0


### 🤔 Q2. 정확도 73%짜리 모델의 정체  



1. 로지스틱 회귀의 정확도는 **0.7303**입니다. 위에서 출력한 **"테스트 세트에서 S의 비율"** 과 나란히 놓고 비교해주세요. 두 숫자가 어떤 관계인가요?
2. `classification_report`에서 클래스 2(C)와 3(Q)의 precision, recall, f1-score가 모두 **0.00**입니다. 이 세 숫자가 동시에 0이 되려면 모델이 무엇을 했어야 하는지 설명해주세요.
3. "모델이 실제로 예측한 값의 분포"와 혼동 행렬을 보고, **이 모델이 실제로 무엇을 하고 있는지** 한 문장으로 설명해주세요.
   > 아래 빈칸을 채우는 형태로 답하면 됩니다.
   > "이 모델은 입력값이 무엇이든 상관없이 `______`."


**답:**
1. 테스트 세트에서의 S비율이 130/178로 로지스틱회귀의 정확도랑 같다. 이는 기존 비율과 모델의 정확도가 같으니 모델의 효과가 없다고 볼 수 있다.
2. precision과 recall이 둘다 0이면 둘이 공유하는 분자인 TP가 0이어야 한다. (f1스코어는 precision과 recall의 조화평균이기에 둘다 0이면 자동으로 0) TP는 정답이 참이고, 참으로 예측한 것이므로 모델이 C와 Q를 한번도 참인 것중에 참이라고 예측한 사람이 없다는 것이다.
3. 이 모델은 입력값이 무엇이든 상관없이 전부 S로 예측한다.


### 🤔 Q3. 왜 이런 일이 생겼을까 

Q3에서 본 현상의 **원인**을 생각해봅시다.

1. 학습 중에 모델이 줄이려고 하는 것은 **"틀린 개수"** 입니다. 훈련 데이터의 클래스 비율(S 72.3% / C 18.6% / Q 9.1%)을 놓고 볼 때, "전부 S라고 답하기"가 모델 입장에서 왜 **합리적인 선택**이었는지 설명해주세요.
2. 이 문제를 개선하려면 어떤 방법을 시도해볼 수 있을까요? 세션에서 배운 내용을 떠올려 **최소 두 가지**를 적고, 각각이 위 1번의 상황을 **어떻게 바꾸는지**도 함께 적어주세요.
   > 💡 여기서 적은 방법을 **Q9에서 실제로 적용해보게 됩니다.** 나중에 다시 돌아와서 비교해보세요.

**답:**
1. C+Q의 비율이 27.7%밖에 안되기 때문에 전부 S라고 하면 맞을 확률이 72.3%나 되기 때문이다.
2. KNN를 사용하면 전체 클래스 비율이 아니라 ㄹ가장 가까운 k개 이웃의 클래스를 보고 예측하기에 이런 현상을 개선할 수 있다. 뿐만 아니라 의사결정나무를 사용하면 데이터를 계속 작은 그룹으로 나눠서 전체에서 소수인 c, q도 특정 그룹 안에서는 다수가 될 수 있어 정확한 에측이 가능할 수 읶다.


## 5. 모델 2, 3, 4 — 의사결정나무 / SVM / kNN

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

models = {
    '로지스틱 회귀': LogisticRegression(),
    '의사결정나무': DecisionTreeClassifier(random_state=42),
    'SVM':         SVC(random_state=42, probability=True),
    'kNN':         KNeighborsClassifier(n_neighbors=5),
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    results.append({
        '모델': name,
        '정확도':      round(accuracy_score(y_test, pred), 4),
        'macro F1':   round(f1_score(y_test, pred, average='macro', zero_division=0), 4),
        'C를 맞힌 수': int(((pred == 2) & (y_test == 2)).sum()),
        'Q를 맞힌 수': int(((pred == 3) & (y_test == 3)).sum()),
    })

print("참고) 무조건 S로 찍기의 정확도: %.4f" % (y_test == 1).mean())
print("참고) 무조건 S로 찍기의 macro F1: %.4f"
      % f1_score(y_test, np.ones(len(y_test), dtype=int), average='macro', zero_division=0))
pd.DataFrame(results).set_index('모델')

참고) 무조건 S로 찍기의 정확도: 0.7303
참고) 무조건 S로 찍기의 macro F1: 0.2814


,정확도,macro F1,C를 맞힌 수,Q를 맞힌 수
모델,,,,
로지스틱 회귀,0.7303,0.2814,0,0
의사결정나무,0.7022,0.4660,8,3
SVM,0.7416,0.4079,0,3
kNN,0.6966,0.4630,7,3


### 🤔 Q4. 정확도와 macro F1이 다른 이야기를 한다  

위 표를 보고 답해주세요.

1. **정확도** 기준으로 1등은 어느 모델인가요? **macro F1** 기준으로는요?
2. 두 지표의 순위가 다릅니다. 이 문제에서는 어느 쪽을 믿어야 할까요? 그 이유는?
3. 정확도가 가장 높은 **SVM(0.7416)** 은 C를 한 명도 못 맞혔고, 정확도가 가장 낮은 **kNN(0.6966)** 은 C를 6명 맞혔습니다. 정확도가 낮은 모델이 더 쓸모 있어 보이는 이 현상을 어떻게 해석해야 할까요?

**답:**
1. SVM이 가장 높다. macro F1 기준으로는 KNN이 가장 높다
2. 정확도가 아닌 macro F1을 기준으로 믿어야 한다. 정답 레이블의 비율이 불균형하기 때문이다. 무조건 S로 찍어도 0.73이나 맞기 때문. 
3. 실제 클래스별 성능을 동일하게 반영하는 macro F1을 함께 보아야 한다... 사실 잘 모르겠습니다....



## 6. 하이퍼파라미터 튜닝 — Grid Search

In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth':         [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 3, 5],
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5, verbose=1, n_jobs=-1)

grid_search.fit(X_train_scaled, y_train)

print("최적 하이퍼파라미터:", grid_search.best_params_)
print("최고 교차검증 정확도: %.4f" % grid_search.best_score_)

best_tree_model = grid_search.best_estimator_

Fitting 5 folds for each of 36 candidates, totalling 180 fits
최적 하이퍼파라미터: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
최고 교차검증 정확도: 0.7243


### 🤔 Q5. Grid Search가 한 일  

1. `param_grid`에 있는 하이퍼파라미터 조합은 **총 몇 개**인가요? 
2. `cv=5`가 붙어 있습니다. 그렇다면 모델은 **총 몇 번 학습**되었을까요?
3. 실행하면 `Fitting 5 folds for each of 36 candidates, totalling 180 fits`라는 메시지가 나옵니다. 2번 답과 일치하나요?
4. `cv=5`는 무슨 뜻이고, 왜 굳이 5번이나 나눠서 검증할까요?

**답:**
1. 4*4*3으로 36개이다
2. 조합당 5번씩 학습한다는 뜻으로 36*5 180번이다
3. 일치한다. 
4. 5는 5분할해서 4개로 트레이닝하고 1개로 검증하는 것으로 한번만 할 때보다 더 정확해진다


In [9]:
tree_pred = best_tree_model.predict(X_test_scaled)

print("튜닝된 Decision Tree 정확도:", accuracy_score(y_test, tree_pred))
print("튜닝 전 Decision Tree 정확도: 0.7022471910112359")
print()
print("튜닝된 Decision Tree macro F1: %.4f"
      % f1_score(y_test, tree_pred, average='macro', zero_division=0))
print("튜닝 전 Decision Tree macro F1: 0.4660")
print()
print(classification_report(y_test, tree_pred, zero_division=0))

튜닝된 Decision Tree 정확도: 0.7696629213483146
튜닝 전 Decision Tree 정확도: 0.7022471910112359

튜닝된 Decision Tree macro F1: 0.5719
튜닝 전 Decision Tree macro F1: 0.4660

              precision    recall  f1-score   support

           1       0.80      0.93      0.86       130
           2       0.63      0.33      0.44        36
           3       0.57      0.33      0.42        12

    accuracy                           0.77       178
   macro avg       0.67      0.53      0.57       178
weighted avg       0.75      0.77      0.74       178



### 🤔 Q6. 튜닝의 효과 읽기  

1. 튜닝 전후로 정확도와 macro F1이 각각 어떻게 변했나요? 
2. 최적 파라미터가 `max_depth=3`으로 나왔습니다. 튜닝 전 기본값은 깊이 제한이 없었는데(`None`), 왜 **얕은 트리**가 더 좋은 결과를 냈을까요?


**답:**

1. 튜닝 이후 둘다 올랐습니다.
2. 깊이 제한이 없으면 완벽하게 분류할 때까지 계속 분류해 과적합이 발생. max_dapth=3이면 큰 특징들을 기반으로 해서 분류해서 더 최적일 수 있음. 



## 7. Random Search

Grid Search가 모든 조합을 빠짐없이 순회한다면, Random Search는 **정해진 횟수만큼 무작위로 뽑아서** 시도합니다.

In [10]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'max_depth':         [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf':  [1, 2, 4, 6, 8],
    'criterion':         ['gini', 'entropy'],
}

random_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=100,
    cv=5, verbose=1, random_state=42, n_jobs=-1)

random_search.fit(X_train_scaled, y_train)

print("최적 하이퍼파라미터:", random_search.best_params_)
print("최고 교차검증 정확도: %.4f" % random_search.best_score_)

best_random_model = random_search.best_estimator_
print("테스트 정확도:", accuracy_score(y_test, best_random_model.predict(X_test_scaled)))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
최적 하이퍼파라미터: {'min_samples_split': 20, 'min_samples_leaf': 2, 'max_depth': 3, 'criterion': 'gini'}
최고 교차검증 정확도: 0.7243
테스트 정확도: 0.7696629213483146


## 8. 만든 모델 사용해보기

튜닝한 모델에 가상의 승객 정보를 넣어 예측을 받아봅니다.

In [11]:
result, probability = predict_embarked(
    best_tree_model, scaler,
    survived=1, pclass=2, sex='female', age=32,
    sibsp=1, parch=2, fare=60, initial='Mrs'
)

print("예측한 탑승 항구:", result)
print("확률:", probability)

예측한 탑승 항구: S
확률: 0.738255033557047


---
## 9. 모델 개선하기

지금까지의 성적표를 정리하면 이렇습니다.

| 모델 | 정확도 | macro F1 |
|---|---|---|
| 무조건 S로 찍기 (기준선) | 0.7303 | 0.2814 |
| 로지스틱 회귀 | 0.7303 | 0.2814 |
| SVM | 0.7416 | 0.4079 |
| kNN | 0.6966 | 0.4544 |
| 의사결정나무 | 0.7022 | 0.4660 |
| **Grid Search로 튜닝한 트리** | **0.7697** | **0.5719** |

정확도는 기준선을 4%p 정도 넘겼고 macro F1도 0.28에서 0.57까지 올라왔습니다. 그래도 **세 항구를 고르게 맞히는 것과는 아직 거리가 멉니다.** Q4에서 적은 개선 방법들을 이제 실제로 적용해봅니다.



### 🤔 Q7. 직접 시도해보기  〔선택 · 자유〕



In [21]:
from sklearn.ensemble import RandomForestClassifier

#소수 클래스 틀리면 더큰 데미지 줌
new_model = RandomForestClassifier(class_weight='balanced', random_state=42)
new_model.fit(X_train_scaled, y_train) # train 데이터 학습
new_pred = new_model.predict(X_test_scaled) #학습한 모델로 test 예측

print("RF 정확도:", accuracy_score(y_test, new_pred)) 
print("RF macro F1:", f1_score(y_test, new_pred, average='macro'))
print(classification_report(y_test, new_pred, zero_division=0))

RF 정확도: 0.6179775280898876
RF macro F1: 0.5352744008206193
              precision    recall  f1-score   support

           1       0.85      0.59      0.70       130
           2       0.42      0.72      0.53        36
           3       0.28      0.58      0.38        12

    accuracy                           0.62       178
   macro avg       0.52      0.63      0.54       178
weighted avg       0.72      0.62      0.64       178

